# 🚀 S21AMF: 1-Click LLM Fine-Tuning Notebook for Google Colab
### Fine-Tune Llama 3.1 (8B) or Qwen 2.5 on Free Google Colab T4 GPU
**Author:** S21AMF Technical Platform  

This notebook trains your custom AI model on technical documents/Q&A, converts it to **GGUF format**, and lets you run it directly in your local **Ollama**!

---  
### Instructions:
1. Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
2. Click **Runtime > Run all** (Ctrl + F9) or run cells one by one.

In [6]:
# Step 1: Install Unsloth and Required Libraries (Fast & Memory Efficient)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
print("✅ Setup Completed!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-co83qp8f/unsloth_f0d3eb80d7cb4b6fabeb7eb3cc5a8a44
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-co83qp8f/unsloth_f0d3eb80d7cb4b6fabeb7eb3cc5a8a44
  Resolved https://github.com/unslothai/unsloth.git to commit a5a992be2da284e9b1623fd04591faf0bea27d8f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 16.4 MB/s eta 0:00:00
   ━━━

In [7]:
# Step 2: Load Llama 3.1 8B (Pre-quantized 4-bit for Free T4 GPU)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Supports up to 8192
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Step 3: Configure LoRA Adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Model Loaded with LoRA Configuration!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model Loaded with LoRA Configuration!


In [8]:
# Step 4: Prepare S21AMF Technical Training Dataset
from datasets import Dataset

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Sample technical data (You can paste as many Q&As as you want here!)
training_data = [
    {
        "instruction": "Calculate the gain difference between dBi and dBd for an amateur antenna.",
        "input": "Given an antenna with 12.15 dBi gain.",
        "output": "The relationship between dBi and dBd is: dBd = dBi - 2.15 dB. Therefore, 12.15 dBi - 2.15 dB = 10.0 dBd gain relative to a half-wave dipole."
    },
    {
        "instruction": "Explain the purpose of a 1:1 Guanella current balun.",
        "input": "Center-fed dipole fed with coaxial cable.",
        "output": "A 1:1 Guanella current balun chokes off common-mode currents on the outside braid of the coaxial transmission line, ensuring clean antenna radiation pattern and preventing RF in the shack."
    },
    {
        "instruction": "What does the center point of a normalized Smith Chart represent?",
        "input": "Normalized transmission line matching.",
        "output": "The center point (1.0 + j0) on a normalized Smith Chart represents a perfect impedance match to the characteristic impedance of the transmission line (e.g. 50 ohms pure resistance, zero reactance, SWR = 1.0:1)."
    }
]

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        text = prompt_template.format(inst, inp, out) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_list(training_data)
dataset = dataset.map(formatting_prompts_func, batched = True)
print("✅ Dataset Ready with", len(dataset), "training examples!")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ Dataset Ready with 3 training examples!


In [9]:
# Step 5: Train the Model (LoRA Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🔥 Starting Training on T4 GPU...")
trainer.train()
print("🎉 Training Completed Successfully!")

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3 [00:00<?, ? examples/s]

🔥 Starting Training on T4 GPU...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,2.412546
2,2.412546
3,2.361429
4,2.176808
5,1.914612
6,1.609509
7,1.246878
8,0.947178
9,0.637727
10,0.498449


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


🎉 Training Completed Successfully!


In [ ]:
# Step 6: Export Directly to GGUF (Ready for Local Ollama)
print("📦 Exporting to GGUF format (Q4_K_M Quantization)...")
model.save_pretrained_gguf("s21amf_custom_model", tokenizer, quantization_method = "q4_k_m")
print("✅ GGUF file generated in ./s21amf_custom_model/")

📦 Exporting to GGUF format (Q4_K_M Quantization)...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in s21amf_custom_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00004-of-00004.safetensors: reconstructing file:   0%|          |  0.00B / 1.17GB            

model-00004-of-00004.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [04:19<00:00, 64.79s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit:  25%|██▌       | 1/4 [01:29<04:29, 89.81s/it]

In [ ]:
# Step 7: Download the Model to your PC or Google Drive
from google.colab import files
import glob

gguf_files = glob.glob("s21amf_custom_model/*.gguf")
if gguf_files:
    print("⬇️ Downloading:", gguf_files[0])
    files.download(gguf_files[0])
else:
    print("Model saved in colab directory.")